# Catalog-level blinding with `desiblind`

**Aim:** demonstrate the `desiblind.catalog.TracerCatalogBlinder` API for catalog-level BAO/AP, RSD, and fNL blinding.

This notebook is `desiblind`-specific. It does not run the `desi-clustering` measurement pipeline; instead, it shows how to generate/load sealed catalog-blinding parameters, inspect sealed provenance, remap redshifts, and call the catalog blinding helpers that downstream pipelines use.

## Environment

Catalog-level blinding uses optional dependencies. Install them with the catalog extra, or run in a DESI environment where they are already available:

```bash
pip install -e /path/to/desiblind[catalog]
```

BAO/AP examples require `cosmoprimo`. RSD/fNL examples additionally require `mockfactory` and prepared catalog columns such as `POSITION` and `INDWEIGHT`.

In [ ]:
from pathlib import Path
import os
import sys

import numpy as np

# If this notebook is run from desiblind/nb, expose the local package checkout.
repo_root = Path.cwd().parent if Path.cwd().name == "nb" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from desiblind.catalog import (
    TracerCatalogBlinder,
    compute_fgrowth_blind,
    generate_catalog_parameters,
)

print(f"Using desiblind checkout: {repo_root}")

## A tiny catalog-like object

`TracerCatalogBlinder` operates on catalog-like objects with column access, assignment, `copy()`, and an `attrs` dictionary. DESI production runs typically pass `mockfactory.Catalog`; this toy class keeps the examples lightweight.

In [ ]:
class ToyCatalog(dict):
    def __init__(self, **columns):
        super().__init__(columns)
        self.attrs = {}
        self.mpicomm = None

    def copy(self):
        new = type(self)(**{key: np.array(value, copy=True) for key, value in self.items()})
        new.attrs = dict(self.attrs)
        new.mpicomm = self.mpicomm
        return new

## Create a toy sealed parameter file

Production analyses should use the approved sealed catalog parameter file for the analysis. The cell below creates a local toy file so the tutorial can be run without exposing or depending on production secrets.

In [ ]:
DEMO_DIR = Path(os.environ.get("TMPDIR", "/tmp")) / "desiblind_catalog_blinding_demo"
DEMO_DIR.mkdir(parents=True, exist_ok=True)

TOY_PARAMETERS_FN = DEMO_DIR / "toy_catalog_blinding_parameters.npy"
TRACER = "LRG"
MODES = ["bao", "rsd", "fnl"]

# Toy values for this notebook only; do not copy them into production validation.
generate_catalog_parameters(
    parameters_fn=TOY_PARAMETERS_FN,
    w0=-0.99,
    wa=0.05,
    fnl_blind=7.0,
    modes=MODES,
    overwrite=True,
    validate=False,
)

print(f"Toy sealed parameter file: {TOY_PARAMETERS_FN}")

The same kind of file can be generated from the command line. For production, use the reviewed parameter source rather than the toy values above:

```bash
desiblind-catalog-parameters \
    --parameters-fn /path/to/<analysis-catalog-blinding-file>.npy \
    --w0 -0.99 --wa 0.05 --fnl-blind 7.0 \
    --modes bao rsd fnl \
    --overwrite
```

A table-driven production generation can instead use `--w0wa-table` and optional `--rows`.

## Load sealed parameters

`load_parameters` selects the deterministic hidden candidate using the existing DESI convention, `RandomState(42).randint(100)`. The returned dictionary contains blind values because it is needed to apply blinding, but downstream notebooks and logs should normally print only sealed attrs.

In [ ]:
params = TracerCatalogBlinder.load_parameters(
    modes=MODES,
    tracer=TRACER,
    parameters_fn=TOY_PARAMETERS_FN,
    metadata="sealed",
    validate=False,
)

sealed_attrs = TracerCatalogBlinder.blinding_attrs(params)
print("Sealed attrs safe for logs/files:")
for key, value in sealed_attrs.items():
    print(f"  {key}: {value}")

## Output suffix helper

Pipelines can use the suffix helper to label products without duplicating mode-normalization logic.

In [ ]:
input_version = "data-dr2-v2"
output_version = TracerCatalogBlinder.output_version(input_version, params)
print(f"{input_version} -> {output_version}")

## BAO/AP redshift remapping

The BAO/AP operation maps redshifts through the blinded `w0`/`wa` cosmology and converts the corresponding distances back through the fiducial DESI distance-redshift relation. Non-finite values are preserved.

In [ ]:
z = np.array([0.4, 0.8, 1.1, np.nan])
z_blinded = TracerCatalogBlinder.transform_redshift(z, w0=params["w0"], wa=params["wa"])
z_roundtrip = TracerCatalogBlinder.transform_redshift(z_blinded, w0=params["w0"], wa=params["wa"], inverse=True)

print("input z:      ", z)
print("blinded z:    ", z_blinded)
print("roundtrip z:  ", z_roundtrip)

## Apply BAO/AP to catalog columns

By default, BAO/AP reads and writes the `Z` column on a copy. For LSS full-catalog workflows, the helper can read `Z_not4clus` and write the blinded values into `Z`.

In [ ]:
catalog = ToyCatalog(
    RA=np.array([10.0, 20.0, 30.0]),
    DEC=np.array([0.0, 5.0, 10.0]),
    Z=np.array([0.41, 0.81, 1.11]),
    Z_not4clus=np.array([0.4, 0.8, 1.1]),
    WEIGHT=np.ones(3),
)

blinded_catalog = TracerCatalogBlinder.apply_bao_blinding(
    catalog,
    params,
    input_zcol="Z_not4clus",
    output_zcol="Z",
)

print("Original Z:       ", catalog["Z"])
print("Source Z_not4clus:", catalog["Z_not4clus"])
print("Blinded Z:        ", blinded_catalog["Z"])
print("Attrs:            ", blinded_catalog.attrs)

## BAO/AP inverse mapping for validation

`remove_bao_blinding` is intentionally BAO-only and requires `force=True`. RSD and fNL catalog-level operations should be treated as apply-only; regenerate from original catalogs instead of trying to reverse them.

In [ ]:
try:
    TracerCatalogBlinder.remove_bao_blinding(blinded_catalog, params)
except ValueError as exc:
    print(exc)

unblinded_catalog = TracerCatalogBlinder.remove_bao_blinding(blinded_catalog, params, force=True)
print("Recovered Z:", unblinded_catalog["Z"])

## RSD and fNL API shape

RSD and fNL blinding operate on prepared catalogs with `POSITION` and `INDWEIGHT`, and require random catalogs. They delegate the heavy lifting to `mockfactory.blinding.CutskyCatalogBlinding`. The cell below is guarded because realistic prepared data/randoms are analysis-specific.

In [ ]:
RUN_RSD_FNL_EXAMPLE = False

if RUN_RSD_FNL_EXAMPLE:
    prepared_data = ...       # mockfactory.Catalog with POSITION, INDWEIGHT, RA, DEC, Z
    prepared_randoms = [...]   # list of matching random catalogs with POSITION, INDWEIGHT

    rsd_data = TracerCatalogBlinder.apply_rsd_blinding(prepared_data, prepared_randoms, params, tracer=TRACER)
    fnl_data, fnl_randoms = TracerCatalogBlinder.apply_fnl_blinding(rsd_data, prepared_randoms, params, tracer=TRACER)
else:
    fgrowth = compute_fgrowth_blind(params["w0"], params["wa"], tracer=TRACER)
    print("RSD/fNL calls skipped. Example fgrowth_blind can be computed internally for the tracer.")
    print(f"Tracer: {TRACER}; modes: {params['modes']}; fgrowth type: {type(fgrowth).__name__}")

## `from_options` for pipeline adapters

Pipeline packages such as `desi-clustering` can pass a compact options dictionary and let `desiblind` normalize modes, load sealed parameters, compute RSD defaults, and build attrs.

In [ ]:
options = dict(
    modes=["ap", "png"],
    parameters_fn=str(TOY_PARAMETERS_FN),
    metadata="sealed",
    validate=False,
)

pipeline_params = TracerCatalogBlinder.from_options(options, tracer="ELG_LOPnotqso")
print("Normalized modes:", pipeline_params["modes"])
print("Output suffix:", TracerCatalogBlinder.output_version("data-dr2-v2", pipeline_params))
print("Sealed attrs:", TracerCatalogBlinder.blinding_attrs(pipeline_params))

## Notes and pitfalls

- Use `metadata="sealed"` by default. `metadata="open"` intentionally exposes blind values and should be reserved for explicit validation checks.
- `bao` and `ap` are aliases; `fnl`, `png`, and `local_png` are aliases.
- The default effective redshift and bias are tracer-family fallbacks. Pass `zeff` and `bias` when an analysis has validated values.
- BAO/AP can be inverted for validation with `force=True`; RSD/fNL are apply-only.
- `TracerCatalogBlinder` keeps heavy imports lazy so data-vector blinding users do not need catalog dependencies unless they use this API.